In [0]:
%python
from databricks.vector_search.client import VectorSearchClient

ENDPOINT_NAME = "sales_tax_vector_endpoint"
INDEX_NAME = "sales_tax_catalog.gold.sales_tax_index"
SOURCE_TABLE = "sales_tax_catalog.gold.sales_tax_rag_context"
EMBEDDING_MODEL = "databricks-bge-large-en"

def sync_vector_search():
    try:
        client = VectorSearchClient()
        
        # 1. Cria ou recupera o endpoint vetorial
        endpoints = [e["name"] for e in client.list_endpoints().get("endpoints", [])]
        if ENDPOINT_NAME not in endpoints:
            print(f"Criando Vector Endpoint {ENDPOINT_NAME}...")
            client.create_endpoint(name=ENDPOINT_NAME, endpoint_type="STANDARD")
            
        # 2. Sincroniza a tabela Gold com o Índice Vetorial
        indexes = [i["name"] for i in client.list_indexes(ENDPOINT_NAME).get("vector_indexes", [])]
        if INDEX_NAME not in indexes:
            print(f"Criando Índice Vetorial {INDEX_NAME}...")
            client.create_delta_sync_index(
                endpoint_name=ENDPOINT_NAME,
                source_table_name=SOURCE_TABLE,
                index_name=INDEX_NAME,
                pipeline_type="TRIGGERED",
                primary_key="id_record",
                embedding_source_column="document_text",
                embedding_model_endpoint_name=EMBEDDING_MODEL
            )
        else:
            print(f"Sincronizando Índice Vetorial {INDEX_NAME}...")
            client.get_index(ENDPOINT_NAME, INDEX_NAME).sync()
            
    except Exception as e:
        print("\n--- AVISO DE AMBIENTE ---")
        print("Este código está correto para ambientes Databricks Premium (Unity Catalog).")
        print("No Databricks Free Edition, o Vector Search não está habilitado.")
        print(f"Erro original: {e}")

if __name__ == "__main__":
    sync_vector_search()